# 6: GCN vs GAT Interpretability Analysis

Comparative analysis of rolling Pearson GCN and 4c GATv2 (Pearson-constrained).
Both models share the same Pearson adjacency mask but differ in edge weights:
- **GCN**: uses raw Pearson correlation values as weights
- **GAT**: learns attention weights within the Pearson mask

**Sections:**
1. GAT Connectivity Analysis (repeat 5b for GAT)
2. Position Analysis across VIX regimes
3. Sector Alignment (Pearson vs learned attention)
4. Attention Asymmetry & Interpretation

## 0. Setup

In [ ]:
import os, sys
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        get_ipython().system('git clone https://github.com/adam-909/4yp.git /content/repo')
    else:
        get_ipython().system('cd /content/repo && git pull')
    os.chdir('/content/repo/4YP-main')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
    from google.colab import drive
    drive.mount('/content/drive')
else:
    os.chdir('/home/adam/new4YP/4YP-main')
    RESULTS_BASE = "FINAL_RESULTS"

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")
print(f"Results base: {RESULTS_BASE}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import networkx as nx
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from gml.experiment_utils import load_experiment_results, compute_graph_stats
from gml.connectivity_analysis import plot_connectivity_vs_performance
from gml.graph_visualization import SECTOR_COLORS
from settings.default import ALL_TICKERS, BBG_SECTORS

# Thesis-quality figure defaults
plt.rcParams.update({
    "font.size": 11,
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.facecolor": "white",
})

print("Imports ready")

In [ ]:
# --- Load all model results ---

GCN_EXPERIMENT = "3_GCN_rolling_pearson"
GCN_CONFIG = "lb20_th0.4_equity"
GCN_SEED = 42

GAT_EXPERIMENT = "4c_GATv2_rolling_pearson_th0.4_scFalse_resTrue"
GAT_CONFIG = "lb20_th0.4_equity"
GAT_SEED = 40

print(f"Loading GCN: {GCN_EXPERIMENT}/{GCN_CONFIG}/seed_{GCN_SEED}")
gcn_results = load_experiment_results(GCN_EXPERIMENT, GCN_SEED, config_name=GCN_CONFIG, base_dir=RESULTS_BASE)

print(f"Loading GAT: {GAT_EXPERIMENT}/{GAT_CONFIG}/seed_{GAT_SEED}")
gat_results = load_experiment_results(GAT_EXPERIMENT, GAT_SEED, config_name=GAT_CONFIG, base_dir=RESULTS_BASE)

# Baselines
print("Loading baselines...")
lstm_results = load_experiment_results("1_lstm_only", 40, base_dir=RESULTS_BASE)
gcn_static_results = load_experiment_results("2_GCN_static", 40, base_dir=RESULTS_BASE)

# Extract key arrays
gcn_adj = gcn_results["adjacency"]            # (W, 88, 88)
gcn_daily = gcn_results["daily_returns"]       # pd.Series
gcn_cr = gcn_results["captured_returns"]       # DataFrame
gcn_dates = gcn_results["test_dates"]          # (W,)

gat_attn = gat_results["attention_weights"]    # (W, 4, 88, 88)
gat_adj = gat_results["adjacency"]             # (W, 88, 88) same Pearson mask
gat_daily = gat_results["daily_returns"]
gat_cr = gat_results["captured_returns"]
gat_dates = gat_results["test_dates"]

lstm_daily = lstm_results["daily_returns"]
gcn_static_daily = gcn_static_results["daily_returns"]

# Head-averaged attention
gat_attn_avg = gat_attn.mean(axis=1)  # (W, 88, 88)

# Ticker / sector setup
tickers = sorted(ALL_TICKERS)
sectors = [BBG_SECTORS.get(t, "Unknown") for t in tickers]
unique_sectors = sorted(set(sectors))
ticker_to_idx = {t: i for i, t in enumerate(tickers)}
sector_to_tickers = {}
for t, s in zip(tickers, sectors):
    sector_to_tickers.setdefault(s, []).append(t)

print(f"\nGCN adjacency: {gcn_adj.shape}")
print(f"GAT attention:  {gat_attn.shape}")
print(f"GAT attn avg:   {gat_attn_avg.shape}")
print(f"Tickers: {len(tickers)}, Sectors: {len(unique_sectors)}")

In [ ]:
# --- Download VIX and define regimes ---
import yfinance as yf

vix = yf.download("^VIX", start="2017-01-01", end="2023-09-01")["Close"].squeeze()
vix.index = pd.to_datetime(vix.index)

gcn_dates_dt = pd.to_datetime(gcn_dates)
gat_dates_dt = pd.to_datetime(gat_dates)
vix_gcn = vix.reindex(gcn_dates_dt, method="ffill")
vix_gat = vix.reindex(gat_dates_dt, method="ffill")

# VIX regime thresholds (quartile-based, model-independent)
vix_q25 = vix_gcn.quantile(0.25)
vix_q75 = vix_gcn.quantile(0.75)
vix_regime_gcn = pd.Series("Mid", index=gcn_dates_dt)
vix_regime_gcn[vix_gcn > vix_q75] = "High"
vix_regime_gcn[vix_gcn < vix_q25] = "Low"

vix_regime_gat = pd.Series("Mid", index=gat_dates_dt)
vix_regime_gat[vix_gat > vix_q75] = "High"
vix_regime_gat[vix_gat < vix_q25] = "Low"

print(f"VIX quartiles: Q25={vix_q25:.1f}, Q75={vix_q75:.1f}")
print(f"GCN regime counts: {vix_regime_gcn.value_counts().to_dict()}")
print(f"GAT regime counts: {vix_regime_gat.value_counts().to_dict()}")

In [ ]:
# --- Shared helper functions ---

def compute_rolling_sharpe(daily_returns, window=20):
    """Annualized rolling Sharpe from daily returns."""
    rolling_mean = daily_returns.rolling(window).mean()
    rolling_std = daily_returns.rolling(window).std()
    return (rolling_mean / rolling_std) * np.sqrt(252)

def compute_edge_counts(adjacencies):
    """Number of undirected edges per window (excluding diagonal)."""
    edge_counts = []
    for adj in adjacencies:
        a = adj.copy()
        np.fill_diagonal(a, 0)
        edge_counts.append((a > 0).sum() / 2)
    return np.array(edge_counts)

def regime_sharpe(daily_returns, regime_series, regime_label):
    """Sharpe ratio for a specific VIX regime."""
    mask = regime_series == regime_label
    regime_rets = daily_returns.reindex(regime_series.index)[mask]
    if regime_rets.std() == 0 or len(regime_rets) < 5:
        return np.nan
    return regime_rets.mean() / regime_rets.std() * np.sqrt(252)

def bootstrap_sharpe_diff(rets_a, rets_b, n_boot=10000):
    """Bootstrap p-value for Sharpe difference between two return series."""
    aligned = pd.concat([rets_a.rename("a"), rets_b.rename("b")], axis=1).dropna()
    diff_obs = aligned["a"].mean() / aligned["a"].std() - aligned["b"].mean() / aligned["b"].std()
    pooled = aligned.values.flatten()
    n = len(aligned)
    diffs = []
    for _ in range(n_boot):
        idx = np.random.choice(len(pooled), size=2*n, replace=True)
        s1, s2 = pooled[idx[:n]], pooled[idx[n:]]
        d = s1.mean() / s1.std() - s2.mean() / s2.std() if s1.std() > 0 and s2.std() > 0 else 0
        diffs.append(d)
    p_val = (np.abs(diffs) >= np.abs(diff_obs)).mean()
    return diff_obs * np.sqrt(252), p_val

print("Helpers defined")

---
## 1. GAT Connectivity Analysis

The GAT uses the same Pearson adjacency mask as GCN but learns edge weights via attention.
We define **effective connectivity** using:
1. **Effective edge count**: edges where head-averaged attention weight > threshold
2. **Mean attention weight**: average learned weight within the Pearson mask

Does GAT's learned connectivity track market stress the same way as Pearson?

In [ ]:
# Compute connectivity metrics for both models
gcn_edges = compute_edge_counts(gcn_adj)

# GAT: effective edges from head-averaged attention
# Also compute mean attention weight within the Pearson mask per window
gat_eff_edges = np.zeros(len(gat_attn_avg))
gat_mean_attn = np.zeros(len(gat_attn_avg))
gcn_mean_weight = np.zeros(len(gcn_adj))

for t in range(len(gat_attn_avg)):
    a = gat_attn_avg[t].copy()
    np.fill_diagonal(a, 0)
    mask = a > 0
    gat_eff_edges[t] = mask.sum() / 2
    gat_mean_attn[t] = a[mask].mean() if mask.any() else 0

    g = gcn_adj[t].copy()
    np.fill_diagonal(g, 0)
    gmask = g > 0
    gcn_mean_weight[t] = g[gmask].mean() if gmask.any() else 0

# Also compute graph stats via utility
gat_graph_stats = compute_graph_stats(gat_attn_avg)

print(f"GCN Pearson edges: mean={gcn_edges.mean():.0f}, range=[{gcn_edges.min():.0f}, {gcn_edges.max():.0f}]")
print(f"GAT effective edges: mean={gat_eff_edges.mean():.0f}, range=[{gat_eff_edges.min():.0f}, {gat_eff_edges.max():.0f}]")
print(f"GCN mean Pearson weight: {gcn_mean_weight.mean():.4f}")
print(f"GAT mean attention weight: {gat_mean_attn.mean():.4f}")

In [ ]:
# Plot: GCN vs GAT connectivity over time with VIX
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

# Top: Edge counts with VIX
ax1 = axes[0]
ax1.plot(gcn_dates_dt, gcn_edges, color="#1f77b4", lw=0.8, label="GCN Pearson edges", alpha=0.8)
ax1.plot(gat_dates_dt, gat_eff_edges, color="#d62728", lw=0.8, label="GAT effective edges", alpha=0.8)
ax1.set_ylabel("Edge Count")
ax1.legend(loc="upper left")

ax1v = ax1.twinx()
ax1v.fill_between(vix_gcn.index, vix_gcn.values, alpha=0.15, color="gray", label="VIX")
ax1v.set_ylabel("VIX", color="gray")
ax1v.tick_params(axis="y", labelcolor="gray")
ax1.set_title("Effective Connectivity: Pearson Edge Count vs GAT Attention Edges")

# Annotate stress events
for ax in [axes[0], axes[1]]:
    ax.axvspan("2020-02-15", "2020-06-15", alpha=0.08, color="red", label="_covid")
    ax.axvspan("2022-01-01", "2022-10-01", alpha=0.08, color="orange", label="_rates")

# Bottom: Mean edge weights
ax2 = axes[1]
ax2.plot(gcn_dates_dt, gcn_mean_weight, color="#1f77b4", lw=0.8, label="GCN mean Pearson weight")
ax2.plot(gat_dates_dt, gat_mean_attn, color="#d62728", lw=0.8, label="GAT mean attention weight")
ax2.set_ylabel("Mean Edge Weight")
ax2.set_xlabel("Date")
ax2.legend(loc="upper left")
ax2.set_title("Mean Edge Weight Within Pearson Mask Over Time")

plt.tight_layout()
plt.show()

In [ ]:
# Concurrent correlation: connectivity vs rolling Sharpe for both models
gcn_rolling_sharpe = compute_rolling_sharpe(gcn_daily.reindex(gcn_dates_dt), window=20)
gat_rolling_sharpe = compute_rolling_sharpe(gat_daily.reindex(gat_dates_dt), window=20)

# Align and correlate
valid_gcn = ~np.isnan(gcn_rolling_sharpe.values)
valid_gat = ~np.isnan(gat_rolling_sharpe.values)

r_gcn, p_gcn = stats.pearsonr(gcn_edges[valid_gcn], gcn_rolling_sharpe.values[valid_gcn])
r_gat, p_gat = stats.pearsonr(gat_eff_edges[valid_gat], gat_rolling_sharpe.values[valid_gat])

# Also correlate mean attention weight with Sharpe
r_gat_wt, p_gat_wt = stats.pearsonr(gat_mean_attn[valid_gat], gat_rolling_sharpe.values[valid_gat])

# VIX correlation
r_gcn_vix, p_gcn_vix = stats.pearsonr(gcn_edges, vix_gcn.values)
r_gat_vix, p_gat_vix = stats.pearsonr(gat_eff_edges[: len(vix_gat)], vix_gat.values[:len(gat_eff_edges)])
r_gat_wt_vix, p_gat_wt_vix = stats.pearsonr(gat_mean_attn[:len(vix_gat)], vix_gat.values[:len(gat_mean_attn)])

print("Connectivity vs Rolling Sharpe (20-window):")
print(f"  GCN edges vs Sharpe:        r={r_gcn:.4f}, p={p_gcn:.4f} {'*' if p_gcn<0.05 else ''}")
print(f"  GAT eff. edges vs Sharpe:   r={r_gat:.4f}, p={p_gat:.4f} {'*' if p_gat<0.05 else ''}")
print(f"  GAT mean attn vs Sharpe:    r={r_gat_wt:.4f}, p={p_gat_wt:.4f} {'*' if p_gat_wt<0.05 else ''}")
print()
print("Connectivity vs VIX:")
print(f"  GCN edges vs VIX:           r={r_gcn_vix:.4f}, p={p_gcn_vix:.2e} {'*' if p_gcn_vix<0.05 else ''}")
print(f"  GAT eff. edges vs VIX:      r={r_gat_vix:.4f}, p={p_gat_vix:.2e} {'*' if p_gat_vix<0.05 else ''}")
print(f"  GAT mean attn vs VIX:       r={r_gat_wt_vix:.4f}, p={p_gat_wt_vix:.2e} {'*' if p_gat_wt_vix<0.05 else ''}")

In [ ]:
# Side-by-side scatter: connectivity vs window returns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GCN
gcn_window_rets = gcn_daily.reindex(gcn_dates_dt).values
ax = axes[0]
ax.scatter(gcn_edges, gcn_window_rets * 100, s=3, alpha=0.3, color="#1f77b4")
z = np.polyfit(gcn_edges, gcn_window_rets * 100, 1)
x_line = np.linspace(gcn_edges.min(), gcn_edges.max(), 100)
ax.plot(x_line, np.polyval(z, x_line), "r--", lw=2)
r, p = stats.pearsonr(gcn_edges, gcn_window_rets)
ax.set_xlabel("Pearson Edge Count")
ax.set_ylabel("Window Return (%)")
ax.set_title(f"GCN: Edges vs Return (r={r:.3f}, p={p:.3f})")

# GAT
gat_window_rets = gat_daily.reindex(gat_dates_dt).values
ax = axes[1]
ax.scatter(gat_mean_attn, gat_window_rets * 100, s=3, alpha=0.3, color="#d62728")
z = np.polyfit(gat_mean_attn, gat_window_rets * 100, 1)
x_line = np.linspace(gat_mean_attn.min(), gat_mean_attn.max(), 100)
ax.plot(x_line, np.polyval(z, x_line), "r--", lw=2)
r, p = stats.pearsonr(gat_mean_attn, gat_window_rets)
ax.set_xlabel("Mean GAT Attention Weight")
ax.set_ylabel("Window Return (%)")
ax.set_title(f"GAT: Mean Attention vs Return (r={r:.3f}, p={p:.3f})")

plt.suptitle("Connectivity vs Performance: Pearson (GCN) vs Attention (GAT)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Regime analysis: Sharpe per VIX regime for all 4 models
models_daily = {
    "LSTM-only": lstm_daily,
    "GCN Static": gcn_static_daily,
    "GCN Rolling": gcn_daily,
    "4c GAT": gat_daily,
}

# Use GCN date index for regime definition (all models aligned to same trading dates)
regime_results = []
for regime in ["High", "Mid", "Low"]:
    row = {"VIX Regime": regime, "Days": (vix_regime_gcn == regime).sum()}
    row["Mean VIX"] = vix_gcn[vix_regime_gcn == regime].mean()
    for model_name, rets in models_daily.items():
        row[model_name] = regime_sharpe(rets, vix_regime_gcn, regime)
    regime_results.append(row)

regime_df = pd.DataFrame(regime_results)
print("Sharpe Ratio by VIX Regime:")
print(regime_df.to_string(index=False, float_format="%.3f"))

# Bootstrap: GCN Rolling vs GAT in each regime
print("\nBootstrap significance (GCN Rolling vs 4c GAT):")
for regime in ["High", "Mid", "Low"]:
    mask = vix_regime_gcn == regime
    gcn_r = gcn_daily.reindex(vix_regime_gcn.index)[mask]
    gat_r = gat_daily.reindex(vix_regime_gcn.index)[mask]
    diff, p = bootstrap_sharpe_diff(gcn_r, gat_r, n_boot=5000)
    print(f"  {regime} VIX: Sharpe diff = {diff:+.3f}, p = {p:.3f} {'*' if p<0.05 else ''}")

In [ ]:
# Bar chart: Sharpe by VIX regime for all models
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(3)
width = 0.18
model_names = ["LSTM-only", "GCN Static", "GCN Rolling", "4c GAT"]
colors = ["#7f7f7f", "#2ca02c", "#1f77b4", "#d62728"]

for i, (model, color) in enumerate(zip(model_names, colors)):
    vals = regime_df[model].values
    ax.bar(x + i * width, vals, width, label=model, color=color, alpha=0.85)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels([f"High VIX\n(>{vix_q75:.0f})", "Mid VIX", f"Low VIX\n(<{vix_q25:.0f})"])
ax.set_ylabel("Sharpe Ratio")
ax.set_title("Model Performance Across VIX Regimes")
ax.legend()
ax.axhline(y=0, color="black", lw=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Densifying vs sparsifying analysis for both models
gcn_edge_diff = np.diff(gcn_edges)
gat_edge_diff = np.diff(gat_eff_edges)

gcn_rets_aligned = gcn_daily.reindex(gcn_dates_dt).values
gat_rets_aligned = gat_daily.reindex(gat_dates_dt).values

results_dir = []
for model_name, edge_diff, rets in [
    ("GCN Rolling", gcn_edge_diff, gcn_rets_aligned[1:]),
    ("4c GAT", gat_edge_diff, gat_rets_aligned[1:]),
]:
    densifying = edge_diff > 0
    sparsifying = edge_diff < 0
    for label, mask in [("Densifying", densifying), ("Sparsifying", sparsifying)]:
        r = rets[mask]
        sharpe = r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan
        results_dir.append({
            "Model": model_name, "Direction": label, "Days": mask.sum(),
            "Sharpe": sharpe, "Ann. Return %": r.mean() * 252 * 100,
        })

dir_df = pd.DataFrame(results_dir)
print("Densifying vs Sparsifying:")
print(dir_df.to_string(index=False, float_format="%.3f"))

# Significance
for model_name, edge_diff, rets in [("GCN", gcn_edge_diff, gcn_rets_aligned[1:]),
                                      ("GAT", gat_edge_diff, gat_rets_aligned[1:])]:
    t, p = stats.mannwhitneyu(rets[edge_diff > 0], rets[edge_diff < 0], alternative="two-sided")
    print(f"\n{model_name} densifying vs sparsifying returns: U-test p={p:.4f} {'*' if p<0.05 else ''}")

---
## 2. Position Analysis: GCN vs GAT Across VIX Regimes

Both models see the same features and graph structure. Differences in positions
reveal what the attention mechanism learns beyond Pearson correlation.

Key question: why did GAT outperform in 2020 (COVID) and 2022 (rate hikes)?

In [ ]:
# Extract per-stock positions for both models
def pivot_positions(cr_df):
    """Pivot captured_returns DataFrame to (dates, tickers) position matrix."""
    df = cr_df.copy()
    df["time"] = pd.to_datetime(df["time"])
    return df.pivot_table(index="time", columns="identifier", values="position", aggfunc="first")

def pivot_captured_returns(cr_df):
    """Pivot to (dates, tickers) captured returns matrix."""
    df = cr_df.copy()
    df["time"] = pd.to_datetime(df["time"])
    return df.pivot_table(index="time", columns="identifier", values="captured_returns", aggfunc="first")

gcn_pos = pivot_positions(gcn_cr)
gat_pos = pivot_positions(gat_cr)
gcn_capret = pivot_captured_returns(gcn_cr)
gat_capret = pivot_captured_returns(gat_cr)

# Align to common dates and tickers
common_dates = gcn_pos.index.intersection(gat_pos.index)
common_tickers = [t for t in tickers if t in gcn_pos.columns and t in gat_pos.columns]
gcn_pos = gcn_pos.loc[common_dates, common_tickers]
gat_pos = gat_pos.loc[common_dates, common_tickers]
gcn_capret = gcn_capret.reindex(index=common_dates, columns=common_tickers)
gat_capret = gat_capret.reindex(index=common_dates, columns=common_tickers)

print(f"Aligned positions: {gcn_pos.shape[0]} dates x {gcn_pos.shape[1]} tickers")
print(f"Date range: {common_dates.min()} to {common_dates.max()}")

In [ ]:
# Per-date cross-sectional position correlation
position_corr = pd.Series(index=common_dates, dtype=float)
for date in common_dates:
    g = gcn_pos.loc[date].values
    a = gat_pos.loc[date].values
    valid = ~np.isnan(g) & ~np.isnan(a)
    if valid.sum() > 10:
        position_corr[date] = stats.pearsonr(g[valid], a[valid])[0]

# Rolling mean for smoother plot
pos_corr_rolling = position_corr.rolling(20).mean()

# Align VIX for overlay
vix_pos = vix.reindex(common_dates, method="ffill")

fig, ax1 = plt.subplots(figsize=(16, 5))
ax1.plot(pos_corr_rolling.index, pos_corr_rolling.values, color="#1f77b4", lw=1.2,
         label="Position correlation (rolling 20d)")
ax1.set_ylabel("Cross-Sectional Position Correlation")
ax1.set_ylim(-0.2, 1.1)

ax2 = ax1.twinx()
ax2.fill_between(vix_pos.index, vix_pos.values, alpha=0.15, color="gray")
ax2.set_ylabel("VIX", color="gray")

ax1.axvspan("2020-02-15", "2020-06-15", alpha=0.08, color="red")
ax1.axvspan("2022-01-01", "2022-10-01", alpha=0.08, color="orange")
ax1.set_title("GCN vs GAT: Cross-Sectional Position Correlation Over Time")
ax1.legend(loc="lower left")
plt.tight_layout()
plt.show()

print(f"Mean position correlation: {position_corr.mean():.3f}")
print(f"Std:  {position_corr.std():.3f}")
print(f"Min:  {position_corr.min():.3f} on {position_corr.idxmin().strftime('%Y-%m-%d')}")
print(f"Max:  {position_corr.max():.3f}")

In [ ]:
# Position statistics by VIX regime
vix_regime_pos = pd.Series("Mid", index=common_dates)
vix_common = vix.reindex(common_dates, method="ffill")
vix_regime_pos[vix_common > vix_q75] = "High"
vix_regime_pos[vix_common < vix_q25] = "Low"

pos_stats = []
for regime in ["High", "Mid", "Low"]:
    mask = vix_regime_pos == regime
    dates_r = common_dates[mask]

    gcn_p = gcn_pos.loc[dates_r]
    gat_p = gat_pos.loc[dates_r]

    # Mean position correlation in this regime
    corrs = position_corr.reindex(dates_r).dropna()

    # Mean absolute position
    gcn_mag = gcn_p.abs().mean().mean()
    gat_mag = gat_p.abs().mean().mean()

    # Turnover: mean |pos_t - pos_{t-1}| per stock per day
    gcn_turn = gcn_p.diff().abs().mean().mean()
    gat_turn = gat_p.diff().abs().mean().mean()

    pos_stats.append({
        "VIX Regime": regime, "Days": mask.sum(),
        "Mean Pos Corr": corrs.mean(),
        "GCN Mean |Pos|": gcn_mag, "GAT Mean |Pos|": gat_mag,
        "GAT/GCN Pos Ratio": gat_mag / gcn_mag if gcn_mag > 0 else np.nan,
        "GCN Turnover": gcn_turn, "GAT Turnover": gat_turn,
    })

pos_stats_df = pd.DataFrame(pos_stats)
print("Position Statistics by VIX Regime:")
print(pos_stats_df.to_string(index=False, float_format="%.4f"))

# t-test: position correlation in high vs low VIX
high_corrs = position_corr[vix_regime_pos == "High"].dropna()
low_corrs = position_corr[vix_regime_pos == "Low"].dropna()
t, p = stats.ttest_ind(high_corrs, low_corrs)
print(f"\nPosition correlation High vs Low VIX: t={t:.3f}, p={p:.4f} {'*' if p<0.05 else ''}")

In [ ]:
# Position size distributions by VIX regime (2x3 grid)
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)

for j, regime in enumerate(["Low", "Mid", "High"]):
    mask = vix_regime_pos == regime
    dates_r = common_dates[mask]

    gcn_vals = gcn_pos.loc[dates_r].values.flatten()
    gat_vals = gat_pos.loc[dates_r].values.flatten()
    gcn_vals = gcn_vals[~np.isnan(gcn_vals)]
    gat_vals = gat_vals[~np.isnan(gat_vals)]

    axes[0, j].hist(gcn_vals, bins=80, density=True, alpha=0.7, color="#1f77b4")
    axes[0, j].axvline(np.mean(np.abs(gcn_vals)), color="red", ls="--", lw=1.5, label=f"mean |pos|={np.mean(np.abs(gcn_vals)):.3f}")
    axes[0, j].set_title(f"GCN — {regime} VIX")
    axes[0, j].legend(fontsize=8)

    axes[1, j].hist(gat_vals, bins=80, density=True, alpha=0.7, color="#d62728")
    axes[1, j].axvline(np.mean(np.abs(gat_vals)), color="blue", ls="--", lw=1.5, label=f"mean |pos|={np.mean(np.abs(gat_vals)):.3f}")
    axes[1, j].set_title(f"GAT — {regime} VIX")
    axes[1, j].legend(fontsize=8)

axes[1, 1].set_xlabel("Position Size")
axes[0, 0].set_ylabel("Density")
axes[1, 0].set_ylabel("Density")
plt.suptitle("Position Size Distributions Across VIX Regimes", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Deep dive: 2020 (COVID) and 2022 (rate hikes)
stress_periods = {
    "COVID (Q1-Q2 2020)": ("2020-01-01", "2020-06-30"),
    "Rate Hikes (Q1-Q3 2022)": ("2022-01-01", "2022-09-30"),
}

stress_results = []
for label, (start, end) in stress_periods.items():
    mask = (common_dates >= start) & (common_dates <= end)
    dates_p = common_dates[mask]
    n_days = len(dates_p)

    gcn_r = gcn_daily.reindex(dates_p).dropna()
    gat_r = gat_daily.reindex(dates_p).dropna()
    lstm_r = lstm_daily.reindex(dates_p).dropna()

    gcn_sharpe = gcn_r.mean() / gcn_r.std() * np.sqrt(252) if gcn_r.std() > 0 else np.nan
    gat_sharpe = gat_r.mean() / gat_r.std() * np.sqrt(252) if gat_r.std() > 0 else np.nan
    lstm_sharpe = lstm_r.mean() / lstm_r.std() * np.sqrt(252) if lstm_r.std() > 0 else np.nan

    corrs = position_corr.reindex(dates_p).dropna()
    gcn_mag = gcn_pos.loc[dates_p].abs().mean().mean()
    gat_mag = gat_pos.loc[dates_p].abs().mean().mean()

    # Paired t-test on daily returns
    aligned = pd.concat([gcn_r.rename("gcn"), gat_r.rename("gat")], axis=1).dropna()
    if len(aligned) > 5:
        t_stat, p_val = stats.ttest_rel(aligned["gat"], aligned["gcn"])
    else:
        t_stat, p_val = np.nan, np.nan

    stress_results.append({
        "Period": label, "Days": n_days,
        "LSTM Sharpe": lstm_sharpe, "GCN Sharpe": gcn_sharpe, "GAT Sharpe": gat_sharpe,
        "GAT - GCN": gat_sharpe - gcn_sharpe,
        "Mean Pos Corr": corrs.mean(),
        "GAT/GCN |Pos|": gat_mag / gcn_mag if gcn_mag > 0 else np.nan,
        "Paired t-test p": p_val,
    })

stress_df = pd.DataFrame(stress_results)
print("Stress Period Deep Dive:")
print(stress_df.to_string(index=False, float_format="%.3f"))